# Roboflow Football Video Detection + Smooth Output + Ellipse Tracking

Bu notebook kendi maç videolarında Roboflow `football-players-detection-3zvbc/11` modelini test etmek için hazırlandı.

Özellikler:
- Video FPS'i korunur, çıktı video kasarak 5/10/15 frame gibi atlamaz.
- Roboflow API her `DETECT_EVERY_N_FRAMES` frame'de çağrılır.
- Aradaki frame'ler de videoya yazılır, böylece çıktı akıcı kalır.
- Oyuncuların altında 2. örnekteki gibi elips/yuvarlak gösterim yapılır.
- Top için üçgen marker çizilir.
- Oyuncu ID'leri sade kutu içinde gösterilir.
- Sonuçlar `.mp4` ve `.csv` olarak kaydedilir.

In [ ]:
!pip -q install inference-sdk supervision opencv-python-headless pandas tqdm

## 1) Ayarlar

İlk test için:
- `DETECT_EVERY_N_FRAMES = 5` hızlıdır ama kutular/ID'ler birkaç frame gecikmeli görünebilir.
- En akıcı ve doğru takip için `DETECT_EVERY_N_FRAMES = 1` yap. Bu daha yavaş ve daha fazla API kullanır.
- Çıktı FPS'i kaynak videodan otomatik alınır.

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import cv2
import json
import math
import time
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode

from inference_sdk import InferenceHTTPClient

# ======================
# MODEL AYARLARI
# ======================
MODEL_ID = "football-players-detection-3zvbc/11"
API_URL = "https://serverless.roboflow.com"

# Roboflow API key'i buraya direkt yazma. Çalıştırınca gizli olarak gir.
ROBOFLOW_API_KEY = getpass("Roboflow API Key: ")

CLIENT = InferenceHTTPClient(
    api_url=API_URL,
    api_key=ROBOFLOW_API_KEY
)

# ======================
# VIDEO / PERFORMANS AYARLARI
# ======================
DETECT_EVERY_N_FRAMES = 5      # 1 = her frame, en iyi ama yavaş. 5 = hızlı test.
MAX_SECONDS = 30               # None yaparsan tüm videoyu işler.
CONFIDENCE_THRESHOLD = 0.25

# Görsel çizim ayarları
SHOW_CLASS_NAME = False        # True yaparsan player/goalkeeper/referee yazar.
SHOW_CONFIDENCE = False        # True yaparsan skor yazar.
SHOW_TRACK_ID = True
DRAW_BOXES = False             # 2. görsele benzemesi için False
DRAW_ELLIPSES = True
DRAW_BALL_TRIANGLE = True

# Çıktı klasörü
OUTPUT_DIR = Path("/content/football_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2) Video yükle

In [ ]:
uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("Video yüklenmedi.")

VIDEO_PATH = list(uploaded.keys())[0]
print("Yüklenen video:", VIDEO_PATH)

## 3) Roboflow sonucunu Detection formatına çeviren yardımcı fonksiyonlar

In [ ]:
def roboflow_predictions_to_xyxy(result, confidence_threshold=0.25):
    """
    Roboflow inference result -> dict list
    """
    preds = result.get("predictions", [])
    detections = []

    for p in preds:
        conf = float(p.get("confidence", 0))
        if conf < confidence_threshold:
            continue

        x = float(p["x"])
        y = float(p["y"])
        w = float(p["width"])
        h = float(p["height"])

        x1 = x - w / 2
        y1 = y - h / 2
        x2 = x + w / 2
        y2 = y + h / 2

        detections.append({
            "xyxy": [x1, y1, x2, y2],
            "confidence": conf,
            "class_name": str(p.get("class", "object"))
        })

    return detections


def class_to_id(class_name):
    mapping = {
        "ball": 0,
        "player": 1,
        "goalkeeper": 2,
        "referee": 3
    }
    return mapping.get(class_name, 99)


def detections_to_supervision(detections):
    import supervision as sv

    if len(detections) == 0:
        return sv.Detections.empty()

    xyxy = np.array([d["xyxy"] for d in detections], dtype=np.float32)
    confidence = np.array([d["confidence"] for d in detections], dtype=np.float32)
    class_id = np.array([class_to_id(d["class_name"]) for d in detections], dtype=int)

    return sv.Detections(
        xyxy=xyxy,
        confidence=confidence,
        class_id=class_id
    )


def get_class_name_from_id(class_id):
    reverse = {
        0: "ball",
        1: "player",
        2: "goalkeeper",
        3: "referee"
    }
    return reverse.get(int(class_id), "object")

## 4) 2. görseldeki gibi elips/yuvarlak çizim fonksiyonları

In [ ]:
def get_color_for_class(class_name):
    """
    OpenCV BGR renkleri.
    İstersen buradan değiştirebilirsin.
    """
    if class_name == "ball":
        return (0, 255, 0)        # yeşil
    if class_name == "referee":
        return (0, 220, 255)      # sarı/turuncu
    if class_name == "goalkeeper":
        return (255, 100, 255)    # mor/pembe
    return (255, 255, 0)          # cyan


def draw_ellipse_under_object(frame, xyxy, color, thickness=3):
    x1, y1, x2, y2 = map(int, xyxy)
    width = max(10, x2 - x1)
    center_x = int((x1 + x2) / 2)
    bottom_y = int(y2)

    # Oyuncu boyutuna göre elips genişliği
    axis_x = max(14, int(width * 0.60))
    axis_y = max(5, int(width * 0.18))

    cv2.ellipse(
        frame,
        center=(center_x, bottom_y),
        axes=(axis_x, axis_y),
        angle=0,
        startAngle=0,
        endAngle=360,
        color=color,
        thickness=thickness,
        lineType=cv2.LINE_AA
    )


def draw_label_box(frame, text, xyxy, color):
    x1, y1, x2, y2 = map(int, xyxy)
    center_x = int((x1 + x2) / 2)
    bottom_y = int(y2)

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.55
    thickness = 2

    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    box_w = tw + 12
    box_h = th + 10

    bx1 = int(center_x - box_w / 2)
    by1 = int(bottom_y + 6)
    bx2 = bx1 + box_w
    by2 = by1 + box_h

    # Görüntü sınırları
    h, w = frame.shape[:2]
    bx1 = max(0, min(bx1, w - box_w))
    bx2 = bx1 + box_w
    by1 = max(0, min(by1, h - box_h))
    by2 = by1 + box_h

    cv2.rectangle(frame, (bx1, by1), (bx2, by2), color, -1)
    cv2.putText(
        frame,
        text,
        (bx1 + 6, by2 - 7),
        font,
        font_scale,
        (0, 0, 0),
        thickness,
        cv2.LINE_AA
    )


def draw_ball_triangle(frame, xyxy, color=(0, 255, 0)):
    x1, y1, x2, y2 = map(int, xyxy)
    center_x = int((x1 + x2) / 2)
    top_y = int(y1)

    pts = np.array([
        [center_x, top_y - 18],
        [center_x - 9, top_y - 36],
        [center_x + 9, top_y - 36]
    ], np.int32)

    cv2.fillPoly(frame, [pts], color)
    cv2.polylines(frame, [pts], True, (0, 0, 0), 2, cv2.LINE_AA)


def draw_detection_view(frame, detections, frame_idx=None, api_time=None):
    """
    Tracking olmadan tek-frame detection görselleştirme.
    """
    out = frame.copy()

    for i, d in enumerate(detections):
        xyxy = d["xyxy"]
        class_name = d["class_name"]
        conf = d["confidence"]
        color = get_color_for_class(class_name)

        if class_name == "ball" and DRAW_BALL_TRIANGLE:
            draw_ball_triangle(out, xyxy, color)
        else:
            if DRAW_ELLIPSES:
                draw_ellipse_under_object(out, xyxy, color)
            if DRAW_BOXES:
                x1, y1, x2, y2 = map(int, xyxy)
                cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)

        label_parts = []
        if SHOW_TRACK_ID:
            label_parts.append(str(i + 1))
        if SHOW_CLASS_NAME:
            label_parts.append(class_name)
        if SHOW_CONFIDENCE:
            label_parts.append(f"{conf:.2f}")

        if label_parts:
            draw_label_box(out, " ".join(label_parts), xyxy, color)

    if frame_idx is not None:
        info = f"frame={frame_idx} detections={len(detections)}"
        if api_time is not None:
            info += f" api={api_time:.2f}s"
        cv2.putText(out, info, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 3, cv2.LINE_AA)

    return out


def draw_tracking_view(frame, tracked_detections):
    """
    Supervision Detections + tracker_id ile 2. görsele benzer çizim.
    """
    out = frame.copy()

    if tracked_detections is None or len(tracked_detections) == 0:
        return out

    xyxys = tracked_detections.xyxy
    class_ids = tracked_detections.class_id
    confidences = tracked_detections.confidence

    tracker_ids = getattr(tracked_detections, "tracker_id", None)
    if tracker_ids is None:
        tracker_ids = [None] * len(xyxys)

    for xyxy, class_id, conf, track_id in zip(xyxys, class_ids, confidences, tracker_ids):
        class_name = get_class_name_from_id(class_id)
        color = get_color_for_class(class_name)

        if class_name == "ball":
            if DRAW_BALL_TRIANGLE:
                draw_ball_triangle(out, xyxy, color)
            if DRAW_BOXES:
                x1, y1, x2, y2 = map(int, xyxy)
                cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        else:
            if DRAW_ELLIPSES:
                draw_ellipse_under_object(out, xyxy, color)
            if DRAW_BOXES:
                x1, y1, x2, y2 = map(int, xyxy)
                cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)

        label_parts = []
        if SHOW_TRACK_ID and track_id is not None:
            label_parts.append(str(int(track_id)))
        if SHOW_CLASS_NAME:
            label_parts.append(class_name)
        if SHOW_CONFIDENCE and conf is not None:
            label_parts.append(f"{float(conf):.2f}")

        if label_parts:
            draw_label_box(out, " ".join(label_parts), xyxy, color)

    return out

## 5) Tek frame testi

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Video açılamadı.")

source_fps = cap.get(cv2.CAP_PROP_FPS)
source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / source_fps if source_fps and source_fps > 0 else None

print("FPS:", source_fps)
print("Boyut:", source_width, "x", source_height)
print("Toplam frame:", total_frames)
print("Süre:", duration)

ret, frame = cap.read()
cap.release()

if not ret:
    raise RuntimeError("İlk frame okunamadı.")

test_frame_path = str(OUTPUT_DIR / "test_frame.jpg")
cv2.imwrite(test_frame_path, frame)

t0 = time.time()
result = CLIENT.infer(test_frame_path, model_id=MODEL_ID)
api_time = time.time() - t0

detections = roboflow_predictions_to_xyxy(result, CONFIDENCE_THRESHOLD)
annotated = draw_detection_view(frame, detections, frame_idx=0, api_time=api_time)

test_output_path = str(OUTPUT_DIR / "test_frame_annotated.jpg")
cv2.imwrite(test_output_path, annotated)

print("Detection sayısı:", len(detections))
print("API süresi:", api_time)

display(HTML(f'<img src="{test_output_path}" width="100%">'))

## 6) Smooth video oluşturma

Önemli:
- Çıktı videosu kaynak videonun FPS'i ile yazılır.
- API sadece `DETECT_EVERY_N_FRAMES` aralığında çağrılır.
- Aradaki frame'ler de videoya eklenir. Bu yüzden çıktı video düşük FPS gibi kasmaz.
- Görsel takip çok gecikiyorsa `DETECT_EVERY_N_FRAMES = 1` yap.

In [ ]:
import supervision as sv

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Video açılamadı.")

source_fps = cap.get(cv2.CAP_PROP_FPS)
if source_fps is None or source_fps <= 0 or math.isnan(source_fps):
    source_fps = 25.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if MAX_SECONDS is None:
    max_frames = total_frames
else:
    max_frames = min(total_frames, int(MAX_SECONDS * source_fps))

print(f"Kaynak FPS: {source_fps}")
print(f"İşlenecek frame: {max_frames}")
print(f"Detection aralığı: Her {DETECT_EVERY_N_FRAMES} frame")

output_video_path = str(OUTPUT_DIR / "football_tracking_ellipse_smooth.mp4")
output_csv_path = str(OUTPUT_DIR / "football_tracking_results.csv")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_video_path, fourcc, source_fps, (width, height))

# ByteTrack ayarı
# Not: fps'i kaynak FPS ile başlatmak ID sürekliliğini iyileştirir.
tracker = sv.ByteTrack(frame_rate=int(round(source_fps)))

last_tracked = sv.Detections.empty()
records = []

temp_frame_path = str(OUTPUT_DIR / "_temp_frame.jpg")

for frame_idx in tqdm(range(max_frames)):
    ret, frame = cap.read()
    if not ret:
        break

    should_detect = (frame_idx % DETECT_EVERY_N_FRAMES == 0)

    if should_detect:
        cv2.imwrite(temp_frame_path, frame)

        t0 = time.time()
        result = CLIENT.infer(temp_frame_path, model_id=MODEL_ID)
        api_time = time.time() - t0

        raw_detections = roboflow_predictions_to_xyxy(result, CONFIDENCE_THRESHOLD)
        sv_detections = detections_to_supervision(raw_detections)

        # ByteTrack güncelle
        last_tracked = tracker.update_with_detections(sv_detections)

        # CSV kayıtları
        tracker_ids = getattr(last_tracked, "tracker_id", None)
        if tracker_ids is None:
            tracker_ids = [None] * len(last_tracked)

        for xyxy, conf, class_id, track_id in zip(
            last_tracked.xyxy,
            last_tracked.confidence,
            last_tracked.class_id,
            tracker_ids
        ):
            x1, y1, x2, y2 = map(float, xyxy)
            records.append({
                "frame": frame_idx,
                "time_sec": frame_idx / source_fps,
                "track_id": None if track_id is None else int(track_id),
                "class_id": int(class_id),
                "class_name": get_class_name_from_id(class_id),
                "confidence": None if conf is None else float(conf),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "center_x": (x1 + x2) / 2,
                "center_y": (y1 + y2) / 2,
                "api_time": api_time
            })

    annotated = draw_tracking_view(frame, last_tracked)

    # İstersen küçük durum yazısı açabilirsin.
    # cv2.putText(annotated, f"frame={frame_idx}", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2, cv2.LINE_AA)

    writer.write(annotated)

cap.release()
writer.release()

df = pd.DataFrame(records)
df.to_csv(output_csv_path, index=False)

print("Video kaydedildi:", output_video_path)
print("CSV kaydedildi:", output_csv_path)
print("Kayıt sayısı:", len(df))

## 7) Colab içinde çıktı videosunu oynat

In [ ]:
def show_video(video_path, width=960):
    mp4 = open(video_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f'''
    <video width="{width}" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    ''')

display(show_video(output_video_path))

## 8) Çıktıları indir

In [ ]:
files.download(output_video_path)
files.download(output_csv_path)

## Sorun giderme

### Video hâlâ kasıyor gibi görünüyorsa
`DETECT_EVERY_N_FRAMES = 1` yap. Bu her frame için Roboflow API çağırır, en doğru ve en akıcı tracking sonucunu verir ama daha yavaş çalışır.

### ID'ler çok değişiyorsa
- Kamera açısı çok değişiyorsa veya oyuncular çok küçükse ID değişimi olabilir.
- `DETECT_EVERY_N_FRAMES = 1` yap.
- Daha uzun analizlerde yerel YOLO modeli + ByteTrack kullanmak daha stabil ve ucuz olur.

### Sadece oyuncuları göstermek istiyorsan
`draw_tracking_view` içinde `class_name == "referee"` veya `class_name == "ball"` olanları atlayacak filtre eklenebilir.